**Nama:** Adhitya Naufal  
**NIM:** 2411512026  

## 1️⃣ The Dining Philosophers Problem

In [ ]:
# Memuat modul untuk menjalankan program lain (di sini skrip Python) sebagai proses terpisah.
import subprocess
# Memuat `sys`; dipakai untuk `sys.executable`, yaitu lokasi interpreter Python yang sedang berjalan.
import sys

# Skrip demo dibangun sebagai satu string: daftar string tiap baris kode, digabung dengan `\n`.
# Alasannya, skrip dijalankan di proses terpisah agar jika deadlock, kernel notebook tidak ikut macet.
skrip_naif = "\n".join([
    # Baris pertama skrip demo: memuat `threading` (Lock/Thread) dan `time` (sleep).
    "import threading, time",
    "",
    # Jumlah filsuf = jumlah garpu = 5.
    "N = 5",
    # Satu `Lock` per garpu (list comprehension, 5 lock). Lock terkunci = garpu sedang dipegang
    # seseorang.
    "forks = [threading.Lock() for _ in range(N)]",
    # Penanda per filsuf apakah sudah berhasil makan; awalnya semua `False`.
    "selesai_makan = [False]*N",
    "",
    # Fungsi yang dijalankan tiap thread; `i` = nomor filsuf (0–4).
    "def filsuf(i):",
    # Garpu kiri filsuf-i adalah garpu dengan indeks yang sama (`i`).
    "    kiri = forks[i]",
    # Garpu kanan adalah garpu tetangga. Operator modulo `% N` membuat meja melingkar (filsuf-4 →
    # garpu 0).
    "    kanan = forks[(i+1) % N]",
    # Mencetak bahwa filsuf akan mengambil garpu kiri.
    "    print(f'Filsuf-{i} mengambil garpu kiri ({i})...')",
    # Mengambil (mengunci) garpu kiri. Jika sedang dipegang orang lain, thread menunggu.
    "    kiri.acquire()",
    # Jeda sengaja agar semua filsuf sempat memegang garpu kiri sebelum ada yang meminta garpu kanan,
    # sehingga deadlock hampir pasti terjadi.
    "    time.sleep(0.2)",
    # Mencetak bahwa filsuf kini meminta garpu kanan.
    "    print(f'Filsuf-{i} mencoba mengambil garpu kanan ({(i+1)%N})...')",
    # Meminta garpu kanan, yang sudah dipegang tetangganya sebagai garpu kiri. Semua filsuf menunggu
    # pada saat yang sama, jadi terbentuk circular wait dan program macet.
    "    kanan.acquire()",
    # Baris ini tidak pernah tercapai saat deadlock; filsuf hanya makan jika memegang dua garpu.
    "    print(f'Filsuf-{i} MAKAN!')",
    # Melepas kedua garpu setelah makan.
    "    kanan.release(); kiri.release()",
    # Menandai filsuf-i sudah makan.
    "    selesai_makan[i] = True",
    "",
    # Membuat 5 thread, satu per filsuf, masing-masing menjalankan `filsuf(i)`.
    "threads = [threading.Thread(target=filsuf, args=(i,)) for i in range(N)]",
    # Menjalankan semua thread hampir bersamaan.
    "for t in threads: t.start()",
    # Menunggu semua thread selesai. Karena deadlock, `join` ini menunggu selamanya.
    "for t in threads: t.join()",
    # Mencetak hasil akhir; baris ini tidak pernah tercapai pada solusi naif.
    "print('Semua filsuf selesai makan:', all(selesai_makan))",
])

# Membuka file sementara untuk ditulisi (mode `"w"`).
with open("/tmp/demo_dining_naif.py", "w") as f:
    # Menulis isi skrip demo ke file tersebut.
    f.write(skrip_naif)

# Awal blok penanganan timeout.
try:
    # Menjalankan skrip sebagai proses baru. `capture_output=True, text=True` menangkap output sebagai
    # teks; `timeout=3` membatasi waktu 3 detik. Jika terlewati, `TimeoutExpired` dilempar.
    hasil = subprocess.run([sys.executable, "/tmp/demo_dining_naif.py"], capture_output=True, text=True, timeout=3)
    # Dijalankan hanya jika skrip selesai normal (tidak terjadi pada kasus ini).
    print("Output:", hasil.stdout)
# Menangkap timeout. Bila skrip belum selesai dalam 3 detik, deadlock dianggap terkonfirmasi.
except subprocess.TimeoutExpired as e:
    # Mencetak pesan bahwa deadlock terkonfirmasi.
    print("TIMEOUT setelah 3 detik -- DEADLOCK TERKONFIRMASI!")
    # Judul untuk output yang sempat tercetak sebelum macet.
    print("Output sebelum macet total:")
    # `e.stdout` berupa bytes sehingga di-`decode()`; jika kosong tercetak `(kosong)`. Terlihat semua
    # filsuf memegang garpu kiri lalu menunggu garpu kanan.
    print(e.stdout.decode() if e.stdout else "(kosong)")

### Solusi 1: Pemutusan Circular Wait via Urutan Asimetris

In [ ]:
# Memuat modul `threading` untuk Thread dan Lock.
import threading
# Memuat `time` untuk `sleep`.
import time

# Jumlah filsuf/garpu.
N = 5
# Satu `Lock` untuk tiap garpu.
forks = [threading.Lock() for _ in range(N)]
# Penghitung berapa kali tiap filsuf makan (awal 0 semua).
selesai_makan = [0] * N

# Fungsi thread; `i` nomor filsuf, `putaran` berapa kali mengulang siklus makan-berpikir.
def filsuf(i, putaran):
    # Garpu kiri = garpu ke-`i`.
    kiri = forks[i]
    # Garpu kanan = tetangga, melingkar dengan modulo.
    kanan = forks[(i + 1) % N]
    # Mengulang siklus sebanyak `putaran` kali (`_` = variabel yang tidak dipakai).
    for _ in range(putaran):
        # PERBAIKAN: filsuf terakhir (i == N-1) mengambil garpu KANAN dulu, baru KIRI
        # Kasus khusus: hanya filsuf terakhir (i = 4) yang memakai urutan terbalik. Inilah pemutus
        # circular wait.
        if i == N - 1:
            # Filsuf terakhir mengambil garpu kanan dulu...
            kanan.acquire()
            # ...baru garpu kiri.
            kiri.acquire()
        # Semua filsuf lain tetap memakai urutan normal.
        else:
            # Ambil garpu kiri dulu...
            kiri.acquire()
            # ...lalu garpu kanan.
            kanan.acquire()
        # Menambah hitungan makan filsuf-i (aman karena tiap filsuf hanya menulis ke indeksnya
        # sendiri).
        selesai_makan[i] += 1  # "makan"
        # Mensimulasikan waktu makan 20 ms selagi memegang dua garpu.
        time.sleep(0.02)
        # Melepas garpu kiri.
        kiri.release()
        # Melepas garpu kanan.
        kanan.release()
        # Mensimulasikan waktu berpikir 10 ms tanpa memegang garpu.
        time.sleep(0.01)  # "berpikir"

# Membuat 5 thread, masing-masing 20 putaran.
threads = [threading.Thread(target=filsuf, args=(i, 20)) for i in range(N)]
# Loop untuk memulai semua thread.
for t in threads:
    # Menjalankan thread.
    t.start()
# Loop untuk menunggu semua thread.
for t in threads:
    # Menunggu thread selesai; kali ini selesai karena tidak ada deadlock.
    t.join()

# Menampilkan hitungan makan tiap filsuf; harus 20 semua (tidak ada yang kelaparan).
print("Jumlah makan per filsuf:", selesai_makan)
# Memverifikasi total = 5 × 20 = 100, sekaligus bukti tidak ada deadlock.
print(f"Total makan: {sum(selesai_makan)} (harus {N*20}) -- TIDAK ADA DEADLOCK")

### Solusi 2: Arbitrator/Waiter (Semaphore Pembatas)

In [ ]:
# Memuat `threading`.
import threading
# Memuat `time`.
import time

# Jumlah filsuf/garpu.
N = 5
# Satu `Lock` per garpu.
forks = [threading.Lock() for _ in range(N)]
# Semaphore dengan nilai awal `N-1 = 4` sebagai arbitrator/pelayan: hanya 4 filsuf yang boleh mencoba
# mengambil garpu bersamaan.
kursi_meja = threading.Semaphore(N - 1)  # arbitrator: maksimal N-1 filsuf boleh coba ambil garpu bersamaan
# Penghitung makan tiap filsuf.
selesai_makan = [0] * N

# Fungsi thread dengan jumlah putaran tertentu.
def filsuf(i, putaran):
    # Garpu kiri.
    kiri = forks[i]
    # Garpu kanan (melingkar).
    kanan = forks[(i + 1) % N]
    # Mengulang siklus makan-berpikir.
    for _ in range(putaran):
        # Meminta izin ke pelayan. Jika 4 filsuf sudah di dalam, filsuf kelima menunggu di sini,
        # sehingga tidak mungkin kelima-limanya memegang garpu kiri sekaligus.
        with kursi_meja:  # cegah SEMUA filsuf mencoba mengambil garpu kiri secara bersamaan
            # Mengambil garpu kiri (semua filsuf memakai urutan yang sama).
            kiri.acquire()
            # Mengambil garpu kanan. Minimal satu filsuf di dalam pasti bisa mendapatkannya, jadi
            # sistem terus maju.
            kanan.acquire()
            # Mencatat satu kali makan.
            selesai_makan[i] += 1
            # Simulasi makan 20 ms.
            time.sleep(0.02)
            # Melepas garpu kanan.
            kanan.release()
            # Melepas garpu kiri. Setelah keluar dari `with`, izin dikembalikan ke semaphore.
            kiri.release()
        # Berpikir 10 ms di luar blok `with` agar filsuf lain dapat masuk.
        time.sleep(0.01)

# Membuat 5 thread, masing-masing 20 putaran.
threads = [threading.Thread(target=filsuf, args=(i, 20)) for i in range(N)]
# Loop memulai thread.
for t in threads:
    # Menjalankan thread.
    t.start()
# Loop menunggu thread.
for t in threads:
    # Menunggu hingga semua filsuf selesai.
    t.join()

# Menampilkan hitungan makan per filsuf.
print("Jumlah makan per filsuf:", selesai_makan)
# Verifikasi total 100 makan tanpa deadlock.
print(f"Total makan: {sum(selesai_makan)} (harus {N*20}) -- TIDAK ADA DEADLOCK")

## 2️⃣ The Readers-Writers Problem

In [ ]:
# Memuat `threading`.
import threading
# Memuat `time`.
import time

# Data yang diperebutkan: dictionary dengan satu nilai. Dictionary dipakai agar bisa diubah dari dalam
# fungsi tanpa `global`.
data_bersama = {"nilai": 0}
# Menghitung berapa reader yang sedang aktif membaca.
read_count = 0
# Lock yang melindungi `read_count` agar update-nya tidak balapan.
mutex_read_count = threading.Lock()
# Lock untuk akses eksklusif; dipegang oleh satu writer atau oleh kelompok reader (diwakili reader
# pertama).
write_lock = threading.Lock()

# Daftar pesan untuk dicetak belakangan, agar urutan kejadian jelas.
log = []
# Lock pelindung `log`.
log_lock = threading.Lock()

# Fungsi pembantu untuk menambah pesan ke log secara thread-safe.
def catat(msg):
    # Mengunci log selama menulis.
    with log_lock:
        # Menambahkan pesan ke akhir log.
        log.append(msg)

# Fungsi thread untuk reader.
def pembaca(id_reader):
    # Mendeklarasikan bahwa `read_count` adalah variabel global yang akan diubah.
    global read_count
    # Bagian masuk: kunci penghitung reader.
    with mutex_read_count:
        # Satu reader baru bergabung.
        read_count += 1
        # Hanya reader pertama yang berurusan dengan `write_lock`.
        if read_count == 1:
            # Reader pertama mengunci writer agar tidak bisa masuk selama ada reader. Reader lain
            # langsung lewat karena tidak menyentuh `write_lock`.
            write_lock.acquire()  # reader PERTAMA mengunci writer
    # Reader membaca dan mencatat nilai saat ini.
    catat(f"Reader-{id_reader} membaca nilai={data_bersama['nilai']}")
    # Simulasi durasi membaca 50 ms.
    time.sleep(0.05)
    # Bagian keluar: kunci penghitung lagi.
    with mutex_read_count:
        # Satu reader selesai.
        read_count -= 1
        # Jika tidak ada reader tersisa...
        if read_count == 0:
            # ...reader terakhir membuka kunci agar writer dapat masuk.
            write_lock.release()  # reader TERAKHIR membuka kembali untuk writer

# Fungsi thread untuk writer.
def penulis(id_writer):
    # Writer mengambil `write_lock`; menunggu sampai tidak ada reader maupun writer lain.
    with write_lock:
        # Mengubah data (+1) dengan akses eksklusif.
        data_bersama["nilai"] += 1
        # Mencatat nilai baru.
        catat(f"Writer-{id_writer} menulis nilai baru={data_bersama['nilai']}")
        # Simulasi durasi menulis 100 ms.
        time.sleep(0.1)

# Membuat 6 thread reader.
threads = [threading.Thread(target=pembaca, args=(i,)) for i in range(6)]
# Menambah 2 thread writer ke daftar.
threads += [threading.Thread(target=penulis, args=(i,)) for i in range(2)]

# Loop memulai semua thread.
for t in threads:
    # Menjalankan thread.
    t.start()
# Loop menunggu semua thread.
for t in threads:
    # Menunggu thread selesai.
    t.join()

# Mengulang seluruh pesan log.
for l in log:
    # Mencetak tiap pesan sesuai urutan kejadian.
    print(l)
# Verifikasi: dua writer masing-masing +1, hasil harus 2.
print(f"\nNilai akhir: {data_bersama['nilai']} (harus 2, karena 2 writer masing-masing +1)")

In [ ]:
# Memuat `threading`.
import threading
# Memuat `time`.
import time

# Penghitung reader aktif.
read_count = 0
# Pelindung `read_count`.
mutex_read_count = threading.Lock()
# Kunci akses eksklusif (dipegang kelompok reader atau satu writer).
write_lock = threading.Lock()
# Bendera berhenti: bila di-`set()`, semua reader keluar dari loop.
berhenti = threading.Event()
# Penghitung total pembacaan; berupa list agar dapat diubah dari dalam fungsi.
jumlah_baca = [0]
# Bendera yang diset begitu writer berhasil masuk; dipakai untuk mendeteksi starvation.
writer_menulis = threading.Event()

# Reader yang membaca berulang-ulang tanpa henti, mensimulasikan banyak reader yang terus berdatangan.
def pembaca_terus_menerus():
    # Menandai `read_count` sebagai global.
    global read_count
    # Mengulang selama belum diperintah berhenti.
    while not berhenti.is_set():
        # Bagian masuk: kunci penghitung.
        with mutex_read_count:
            # Reader bergabung.
            read_count += 1
            # Hanya reader pertama yang mengunci writer.
            if read_count == 1:
                # Reader pertama mengunci `write_lock`.
                write_lock.acquire()
        # Menghitung satu pembacaan.
        jumlah_baca[0] += 1
        # Membaca 5 ms. Karena 4 reader bergantian dan saling tumpang tindih, `read_count` hampir
        # tidak pernah turun ke 0.
        time.sleep(0.005)
        # Bagian keluar: kunci penghitung.
        with mutex_read_count:
            # Reader selesai.
            read_count -= 1
            # Hanya bila reader terakhir keluar...
            if read_count == 0:
                # ...`write_lock` dilepas. Kondisi ini hampir tidak pernah terpenuhi sehingga writer
                # tidak pernah kebagian.
                write_lock.release()

# Satu writer yang ingin menulis.
def satu_penulis():
    # Mencatat waktu mulai menunggu.
    t0 = time.perf_counter()
    # Writer mencoba mengambil `write_lock`; terblokir selama ada reader aktif.
    with write_lock:
        # Menandai bahwa writer akhirnya berhasil masuk.
        writer_menulis.set()
    # Mencetak berapa lama writer menunggu.
    print(f"Writer akhirnya dapat giliran setelah {time.perf_counter()-t0:.2f} detik menunggu")

# Membuat 4 thread reader terus-menerus.
readers = [threading.Thread(target=pembaca_terus_menerus) for _ in range(4)]
# Loop memulai reader.
for t in readers:
    # Menjalankan reader.
    t.start()
# Memberi waktu agar reader ramai lebih dulu.
time.sleep(0.1)  # biarkan reader ramai duluan
# Membuat thread writer.
writer = threading.Thread(target=satu_penulis)
# Menjalankan writer yang kemudian terblokir.
writer.start()
# Menunggu writer maksimal 2 detik; tanpa `timeout`, program bisa menunggu selamanya.
writer.join(timeout=2)
# Jika setelah 2 detik writer belum masuk...
if not writer_menulis.is_set():
    # ...cetak bukti bahwa terjadi starvation.
    print("Writer BELUM juga dapat giliran setelah 2 detik -- STARVATION TERBUKTI!")
# Memerintahkan semua reader berhenti agar program bisa selesai.
berhenti.set()
# Loop menunggu reader.
for t in readers:
    # Menunggu reader selesai; setelah itu `write_lock` terlepas dan writer dapat masuk.
    t.join()
# Menampilkan banyaknya pembacaan yang terjadi selama writer menunggu.
print("Total pembacaan yang terjadi selama writer menunggu:", jumlah_baca[0])

### Solusi Fair (Mencegah Writer Starvation)

In [ ]:
# Memuat `threading`.
import threading
# Memuat `time`.
import time

# Penghitung reader aktif.
read_count = 0
# Pelindung `read_count`.
mutex_read_count = threading.Lock()
# Kunci akses eksklusif.
write_lock = threading.Lock()
# Perbaikan: satu lock sebagai pintu masuk bersama reader dan writer; siapa yang lebih dulu antre di
# sini dilayani lebih dulu.
antrian_masuk = threading.Lock()  # PERBAIKAN: pintu masuk tambahan bersama reader & writer
# Bendera berhenti untuk reader.
berhenti = threading.Event()
# Penghitung pembacaan.
jumlah_baca = [0]
# Bendera writer berhasil masuk.
writer_menulis = threading.Event()

# Reader yang membaca terus-menerus.
def pembaca_terus_menerus():
    # Menandai `read_count` sebagai global.
    global read_count
    # Loop sampai dihentikan.
    while not berhenti.is_set():
        # Reader baru harus lewat pintu `antrian_masuk`. Jika writer sudah memegangnya (menunggu
        # `write_lock`), reader baru tertahan di sini.
        with antrian_masuk:  # reader BARU harus antre di sini juga, sama seperti writer
            # Mengunci penghitung saat masuk.
            with mutex_read_count:
                # Reader bergabung.
                read_count += 1
                # Hanya reader pertama yang mengunci writer.
                if read_count == 1:
                    # Mengunci `write_lock`.
                    write_lock.acquire()
        # Menghitung pembacaan (di luar pintu masuk, sehingga banyak reader bisa membaca bersamaan).
        jumlah_baca[0] += 1
        # Membaca 5 ms.
        time.sleep(0.005)
        # Mengunci penghitung saat keluar.
        with mutex_read_count:
            # Reader selesai.
            read_count -= 1
            # Jika reader terakhir...
            if read_count == 0:
                # ...melepas `write_lock`. Kini reader baru tertahan di `antrian_masuk`, jadi kondisi
                # ini pasti tercapai.
                write_lock.release()

# Satu writer.
def satu_penulis():
    # Mencatat waktu mulai menunggu.
    t0 = time.perf_counter()
    # Writer antre di pintu yang sama dengan reader. Selama writer di sini, tidak ada reader baru yang
    # bisa masuk.
    with antrian_masuk:  # writer antre di PINTU YANG SAMA dengan reader -> adil
        # Menunggu reader yang sudah di dalam selesai, lalu mengambil kunci eksklusif.
        with write_lock:
            # Menandai writer berhasil masuk.
            writer_menulis.set()
    # Mencetak lama menunggu (kini sangat singkat).
    print(f"Writer akhirnya dapat giliran setelah {time.perf_counter()-t0:.2f} detik menunggu")

# Membuat 4 reader terus-menerus.
readers = [threading.Thread(target=pembaca_terus_menerus) for _ in range(4)]
# Loop memulai reader.
for t in readers:
    # Menjalankan reader.
    t.start()
# Memberi waktu reader ramai lebih dulu.
time.sleep(0.1)
# Membuat thread writer.
writer = threading.Thread(target=satu_penulis)
# Menjalankan writer.
writer.start()
# Menunggu writer maksimal 2 detik.
writer.join(timeout=2)
# Cek apakah writer masih kelaparan.
if not writer_menulis.is_set():
    # Hanya tercetak jika starvation masih terjadi (tidak terjadi di sini).
    print("Writer masih starvation!")
# Menghentikan semua reader.
berhenti.set()
# Loop menunggu reader.
for t in readers:
    # Menunggu reader selesai.
    t.join()
# Jumlah pembacaan selama writer menunggu, jauh lebih sedikit dibanding solusi reader-preference
# karena reader baru ditahan.
print("Total pembacaan yang terjadi selama writer menunggu:", jumlah_baca[0])

## 3️⃣ The Producer-Consumer Problem: 3 Implementasi

In [ ]:
# Memuat `threading`.
import threading
# Memuat `queue`, berisi `queue.Queue` yang thread-safe.
import queue
# Memuat `time`.
import time

# Producer: menerima antrean `q` dan jumlah item yang dihasilkan.
def producer_thread(q, jumlah):
    # Menghasilkan angka 1 sampai `jumlah`.
    for i in range(1, jumlah + 1):
        # Simulasi waktu produksi 50 ms per item.
        time.sleep(0.05)
        # Memasukkan item ke antrean (thread-safe tanpa lock manual).
        q.put(i)
        # Mencetak item yang baru dihasilkan.
        print(f"[Threading] Producer menghasilkan: {i}")
    # Menaruh `None` sebagai sentinel (tanda selesai) untuk consumer.
    q.put(None)

# Consumer: menerima antrean dan list tempat menyimpan hasil.
def consumer_thread(q, hasil):
    # Akumulator jumlah.
    total = 0
    # Loop sampai sentinel diterima.
    while True:
        # Mengambil item; memblokir bila antrean kosong sampai ada item.
        item = q.get()
        # Cek sentinel.
        if item is None:
            # Keluar dari loop saat produksi selesai.
            break
        # Menambahkan item ke jumlah.
        total += item
    # Menyimpan total ke list (thread tidak bisa `return` ke pemanggil).
    hasil.append(total)

# Membuat antrean FIFO tanpa batas ukuran.
q_thread = queue.Queue()
# List penampung hasil.
hasil_thread = []

# Mencatat waktu mulai dengan timer beresolusi tinggi.
start = time.perf_counter()
# Membuat thread producer dengan argumen (antrean, 6).
t_p = threading.Thread(target=producer_thread, args=(q_thread, 6))
# Membuat thread consumer dengan argumen (antrean, list hasil).
t_c = threading.Thread(target=consumer_thread, args=(q_thread, hasil_thread))
# Menjalankan keduanya; consumer langsung menunggu item dari producer.
t_p.start(); t_c.start()
# Menunggu keduanya selesai.
t_p.join(); t_c.join()
# Menghitung durasi total.
waktu_thread = time.perf_counter() - start

# Verifikasi: 1+2+…+6 = 21.
print(f"\n[Threading] Total hasil consumer: {hasil_thread[0]} (harus {sum(range(1,7))})")
# Menampilkan waktu; sekitar 6 × 0,05 = 0,30 detik karena producer yang menentukan.
print(f"[Threading] Waktu: {waktu_thread:.2f} detik")

### 3.2 `multiprocessing.Queue` (Multiprocessing)

In [ ]:
# Memuat `multiprocessing` dengan alias `mp`.
import multiprocessing as mp
# Memuat `time`.
import time

# Producer yang berjalan di proses terpisah.
def producer_proses(q, jumlah):
    # Menghasilkan angka 1 sampai `jumlah`.
    for i in range(1, jumlah + 1):
        # Simulasi produksi 50 ms.
        time.sleep(0.05)
        # Mengirim item lewat `mp.Queue` (item di-*pickle* dan dikirim antar-proses).
        q.put(i)
        # Mencetak item yang dihasilkan.
        print(f"[Multiprocessing] Producer menghasilkan: {i}")
    # Sentinel penanda selesai.
    q.put(None)

# Consumer di proses lain; hasil dikirim lewat antrean kedua.
def consumer_proses(q, hasil_queue):
    # Akumulator.
    total = 0
    # Loop sampai sentinel.
    while True:
        # Mengambil item; memblokir bila kosong.
        item = q.get()
        # Cek sentinel.
        if item is None:
            # Keluar dari loop.
            break
        # Menambah ke jumlah.
        total += item
    # Mengirim hasil ke proses induk. Memori antar-proses tidak dibagi, jadi `list.append` seperti
    # pada thread tidak akan terlihat induk.
    hasil_queue.put(total)

# Wajib pada multiprocessing, terutama di Windows/macOS (metode *spawn*), agar proses anak tidak
# mengeksekusi ulang kode pembuat proses.
if __name__ == "__main__":
    # Antrean antar-proses untuk item.
    q_proses = mp.Queue()
    # Antrean antar-proses untuk hasil.
    hasil_queue = mp.Queue()

    # Mencatat waktu mulai.
    start = time.perf_counter()
    # Membuat proses producer.
    p_producer = mp.Process(target=producer_proses, args=(q_proses, 6))
    # Membuat proses consumer.
    p_consumer = mp.Process(target=consumer_proses, args=(q_proses, hasil_queue))
    # Menjalankan kedua proses OS.
    p_producer.start(); p_consumer.start()
    # Menunggu kedua proses selesai.
    p_producer.join(); p_consumer.join()
    # Menghitung durasi total.
    waktu_proses = time.perf_counter() - start

    # Mengambil hasil dari antrean hasil.
    hasil_mp = hasil_queue.get()
    # Verifikasi total = 21.
    print(f"\n[Multiprocessing] Total hasil consumer: {hasil_mp} (harus {sum(range(1,7))})")
    # Menampilkan waktu; sedikit lebih besar dari versi thread karena biaya membuat proses dan
    # pickling.
    print(f"[Multiprocessing] Waktu: {waktu_proses:.2f} detik")

### 3.3 `asyncio.Queue` (Asyncio)

In [ ]:
# Memuat `asyncio`.
import asyncio
# Memuat `time`.
import time

# Coroutine producer (`async def`).
async def producer_async(q, jumlah):
    # Menghasilkan angka 1 sampai `jumlah`.
    for i in range(1, jumlah + 1):
        # Tidur non-blocking 50 ms; event loop bisa menjalankan coroutine lain.
        await asyncio.sleep(0.05)
        # Memasukkan item; `await` karena antrean bisa penuh bila memakai `maxsize`.
        await q.put(i)
        # Mencetak item yang dihasilkan.
        print(f"[Asyncio] Producer menghasilkan: {i}")
    # Sentinel selesai.
    await q.put(None)

# Coroutine consumer.
async def consumer_async(q):
    # Akumulator.
    total = 0
    # Loop sampai sentinel.
    while True:
        # Mengambil item; bila kosong, coroutine menyerahkan kendali ke event loop.
        item = await q.get()
        # Cek sentinel.
        if item is None:
            # Keluar dari loop.
            break
        # Menambah ke jumlah.
        total += item
    # Coroutine boleh langsung `return` hasil.
    return total

# Antrean khusus asyncio (bukan thread-safe, tetapi aman karena hanya satu thread).
q_async = asyncio.Queue()

# Mencatat waktu mulai.
start = time.perf_counter()
# `gather` menjalankan dua coroutine bersamaan dan mengembalikan daftar hasil berurutan; hasil
# producer (`None`) dibuang ke `_`, hasil consumer disimpan. `await` di level atas hanya bisa dipakai
# di Jupyter.
_, hasil_async = await asyncio.gather(
    # Coroutine producer dengan 6 item.
    producer_async(q_async, 6),
    # Coroutine consumer.
    consumer_async(q_async),
)
# Menghitung durasi total.
waktu_async = time.perf_counter() - start

# Verifikasi total = 21.
print(f"\n[Asyncio] Total hasil consumer: {hasil_async} (harus {sum(range(1,7))})")
# Menampilkan waktu; sekitar 0,30 detik dengan overhead paling kecil.
print(f"[Asyncio] Waktu: {waktu_async:.2f} detik")

## 4️⃣ The Sleeping Barber Problem

In [ ]:
# Memuat `threading`.
import threading
# Memuat `time`.
import time
# Memuat `random` untuk jeda kedatangan acak.
import random

# Kapasitas ruang tunggu: 3 kursi.
KURSI_TUNGGU = 3

# Semaphore (awal 0) yang menghitung pelanggan menunggu. Saat 0, tukang cukur yang memanggil
# `acquire()` tidur; `release()` dari pelanggan membangunkannya.
pelanggan_menunggu = threading.Semaphore(0)   # jumlah pelanggan menunggu (membangunkan tukang cukur)
# Semaphore (awal 0) sinyal dari tukang cukur: "saya siap melayani".
tukang_cukur_siap = threading.Semaphore(0)    # tukang cukur siap melayani (membangunkan pelanggan tertentu)
# Semaphore (awal 0) sinyal dari tukang cukur: "cukuran selesai".
selesai_dilayani = threading.Semaphore(0)     # menandai satu sesi cukur selesai
# Lock pelindung `jumlah_menunggu`.
mutex = threading.Lock()
# Jumlah pelanggan yang sedang duduk menunggu (list agar bisa diubah di dalam fungsi).
jumlah_menunggu = [0]

# Lock agar pesan `print` dari banyak thread tidak bercampur.
log_lock = threading.Lock()
# Fungsi cetak thread-safe.
def catat(msg):
    # Mengunci selama mencetak.
    with log_lock:
        # Mencetak pesan.
        print(msg)

# Fungsi thread tukang cukur.
def tukang_cukur():
    # Tukang cukur bekerja terus tanpa henti.
    while True:
        # Mencatat bahwa ia tidur menunggu pelanggan.
        catat("[Tukang cukur] tidur, menunggu pelanggan...")
        # Memblokir sampai ada pelanggan (nilai semaphore > 0); inilah "tidur".
        pelanggan_menunggu.acquire()  # tidur sampai ada pelanggan datang
        # Mengunci data ruang tunggu.
        with mutex:
            # Satu pelanggan pindah dari kursi tunggu ke kursi cukur, sehingga satu kursi tunggu
            # kosong.
            jumlah_menunggu[0] -= 1
        # Mencatat mulai mencukur.
        catat("[Tukang cukur] mencukur pelanggan...")
        # Memberi tahu pelanggan bahwa tukang cukur siap.
        tukang_cukur_siap.release()   # beri tahu pelanggan: silakan duduk, saya siap
        # Simulasi waktu cukur 150 ms.
        time.sleep(0.15)              # simulasi waktu potong rambut
        # Mencatat selesai.
        catat("[Tukang cukur] selesai mencukur satu pelanggan")
        # Memberi tahu pelanggan bahwa cukuran selesai.
        selesai_dilayani.release()

# Fungsi thread pelanggan.
def pelanggan(id_pelanggan):
    # Mengunci agar pengecekan dan perubahan jumlah kursi atomik.
    with mutex:
        # Masih ada kursi kosong?
        if jumlah_menunggu[0] < KURSI_TUNGGU:
            # Ya: pelanggan duduk, jumlah menunggu bertambah.
            jumlah_menunggu[0] += 1
            # Mencatat duduk dan jumlah kursi terpakai.
            catat(f"[Pelanggan-{id_pelanggan}] duduk di ruang tunggu ({jumlah_menunggu[0]}/{KURSI_TUNGGU})")
            # Menandai pelanggan ini akan menunggu dilayani.
            harus_tunggu = True
        # Jika semua kursi terisi...
        else:
            # ...pelanggan pergi (penolakan/*rejection*).
            catat(f"[Pelanggan-{id_pelanggan}] ruang tunggu PENUH, pergi!")
            # Menandai pelanggan ini tidak akan dilayani.
            harus_tunggu = False
    # Hanya pelanggan yang duduk yang menjalankan langkah berikut. Dilakukan di luar `mutex` agar
    # tidak menahan lock sambil menunggu.
    if harus_tunggu:
        # Menambah hitungan pelanggan menunggu sekaligus membangunkan tukang cukur jika sedang tidur.
        pelanggan_menunggu.release()  # bangunkan tukang cukur / masuk antrean
        # Menunggu sampai tukang cukur siap.
        tukang_cukur_siap.acquire()   # tunggu sampai tukang cukur siap untukku
        # Menunggu sampai selesai dicukur.
        selesai_dilayani.acquire()    # tunggu sampai selesai dicukur

# Membuat thread tukang cukur sebagai `daemon` agar otomatis berhenti saat program utama selesai
# (karena loop-nya tak berujung).
t_cukur = threading.Thread(target=tukang_cukur, daemon=True)
# Menjalankan tukang cukur.
t_cukur.start()

# Mengunci seed agar pola kedatangan bisa diulang.
random.seed(1)
# Daftar thread pelanggan untuk di-`join`.
pelanggan_threads = []
# Membuat 8 pelanggan.
for i in range(8):
    # Jeda acak 20–100 ms antar kedatangan.
    time.sleep(random.uniform(0.02, 0.1))  # pelanggan datang satu-satu dengan jeda acak
    # Membuat thread pelanggan ke-`i`.
    t = threading.Thread(target=pelanggan, args=(i,))
    # Menjalankan thread pelanggan.
    t.start()
    # Menyimpan thread.
    pelanggan_threads.append(t)

# Loop menunggu semua pelanggan.
for t in pelanggan_threads:
    # Menunggu pelanggan selesai (dicukur atau pergi).
    t.join()

# Pesan penutup.
print("\nSemua pelanggan sudah ditangani (dilayani atau pergi karena penuh).")

**Perhatikan:** pada output, ketika ruang tunggu (`KURSI_TUNGGU=3`) penuh, pelanggan berikutnya **langsung pergi** tanpa menunggu — persis seperti server yang menolak koneksi baru saat connection pool penuh, bukan membuat klien menunggu tanpa batas.

## 5️⃣ The Cigarette Smokers Problem

In [ ]:
# Memuat `subprocess` untuk menjalankan skrip di proses terpisah.
import subprocess
# Memuat `sys` untuk `sys.executable`.
import sys

# Skrip demo disusun sebagai string; dijalankan di proses lain agar macetnya tidak mengganggu kernel.
skrip_naif = "\n".join([
    # Memuat modul yang dibutuhkan skrip demo.
    "import threading, random, time",
    "",
    # Semaphore agent (awal 1): agent boleh menaruh bahan hanya bila perokok sebelumnya selesai.
    "agent_sem = threading.Semaphore(1)",
    # Sinyal "ada tembakau di meja".
    "tobacco_sem = threading.Semaphore(0)",
    # Sinyal "ada kertas di meja".
    "paper_sem = threading.Semaphore(0)",
    # Sinyal "ada korek di meja".
    "match_sem = threading.Semaphore(0)",
    # Sinyal untuk perokok yang punya korek (karena meja berisi tembakau + kertas).
    "tobaccoPaper_sem = threading.Semaphore(0)",
    # Sinyal untuk perokok yang punya tembakau (meja: kertas + korek).
    "paperMatch_sem = threading.Semaphore(0)",
    # Sinyal untuk perokok yang punya kertas (meja: tembakau + korek).
    "tobaccoMatch_sem = threading.Semaphore(0)",
    # Penghitung berapa rokok sudah dihisap.
    "putaran_selesai = [0]",
    # Target jumlah putaran.
    "TARGET = 6",
    "",
    # Fungsi agent.
    "def agent():",
    # Agent mengulang sebanyak `TARGET` kali.
    "    for _ in range(TARGET):",
    # Menunggu giliran; dirilis oleh perokok yang selesai.
    "        agent_sem.acquire()",
    # Memilih dua bahan acak dari tiga kemungkinan pasangan.
    "        pilihan = random.choice(['tobacco_paper', 'paper_match', 'tobacco_match'])",
    # Mencetak pasangan yang ditaruh.
    "        print(f'[Agent] menaruh: {pilihan}')",
    # Jika tembakau + kertas...
    "        if pilihan == 'tobacco_paper':",
    # ...rilis sinyal tembakau dan kertas.
    "            tobacco_sem.release(); paper_sem.release()",
    # Jika kertas + korek...
    "        elif pilihan == 'paper_match':",
    # ...rilis sinyal kertas dan korek.
    "            paper_sem.release(); match_sem.release()",
    # Selain itu (tembakau + korek)...
    "        else:",
    # ...rilis sinyal tembakau dan korek.
    "            tobacco_sem.release(); match_sem.release()",
    "",
    # Pusher untuk pasangan tembakau + kertas.
    "def pusher_tp():",
    # Pusher berulang sampai target tercapai.
    "    while putaran_selesai[0] < TARGET:",
    # Mengambil sinyal tembakau dan kertas satu per satu. Di sinilah race condition terjadi: pusher
    # bisa mendapat satu sinyal lalu terjebak menunggu sinyal lain yang tidak datang.
    "        tobacco_sem.acquire(); paper_sem.acquire()",
    # Membangunkan perokok yang punya korek.
    "        tobaccoPaper_sem.release()",
    "",
    # Pusher untuk pasangan kertas + korek.
    "def pusher_pm():",
    # Loop sampai target.
    "    while putaran_selesai[0] < TARGET:",
    # Mengambil sinyal kertas dan korek.
    "        paper_sem.acquire(); match_sem.acquire()",
    # Membangunkan perokok yang punya tembakau.
    "        paperMatch_sem.release()",
    "",
    # Pusher untuk pasangan tembakau + korek.
    "def pusher_tm():",
    # Loop sampai target.
    "    while putaran_selesai[0] < TARGET:",
    # Mengambil sinyal tembakau dan korek. Pusher ini berebut `tobacco_sem` dengan `pusher_tp`.
    "        tobacco_sem.acquire(); match_sem.acquire()",
    # Membangunkan perokok yang punya kertas.
    "        tobaccoMatch_sem.release()",
    "",
    # Fungsi perokok dengan nama dan semaphore sinyalnya.
    "def perokok(nama, sem):",
    # Loop sampai target.
    "    while putaran_selesai[0] < TARGET:",
    # Menunggu sinyal bahwa bahan yang dibutuhkan sudah lengkap.
    "        sem.acquire()",
    # Mencetak bahwa perokok menghisap.
    "        print(f'  [{nama}] menghisap rokok')",
    # Menambah hitungan putaran.
    "        putaran_selesai[0] += 1",
    # Memberi giliran agent menaruh bahan berikutnya.
    "        agent_sem.release()",
    "",
    # Awal daftar thread: agent, tiga pusher, tiga perokok.
    "threads = [",
    # Thread agent, `daemon` agar tidak menghalangi program berhenti.
    "    threading.Thread(target=agent, daemon=True),",
    # Thread pusher tembakau + kertas.
    "    threading.Thread(target=pusher_tp, daemon=True),",
    # Thread pusher kertas + korek.
    "    threading.Thread(target=pusher_pm, daemon=True),",
    # Thread pusher tembakau + korek.
    "    threading.Thread(target=pusher_tm, daemon=True),",
    # Perokok yang punya korek (menunggu sinyal tembakau+kertas).
    "    threading.Thread(target=perokok, args=('Perokok-Korek', tobaccoPaper_sem), daemon=True),",
    # Perokok yang punya tembakau (menunggu sinyal kertas+korek).
    "    threading.Thread(target=perokok, args=('Perokok-Tembakau', paperMatch_sem), daemon=True),",
    # Perokok yang punya kertas (menunggu sinyal tembakau+korek).
    "    threading.Thread(target=perokok, args=('Perokok-Kertas', tobaccoMatch_sem), daemon=True),",
    "]",
    # Menjalankan semua thread.
    "for t in threads: t.start()",
    # Menunggu semua thread; macet karena ada pusher yang terjebak.
    "for t in threads: t.join()",
    # Hasil akhir; tidak tercapai jika macet.
    "print('Putaran selesai:', putaran_selesai[0], '/', TARGET)",
])

# Membuka file sementara untuk ditulisi.
with open("/tmp/demo_smokers_naif.py", "w") as f:
    # Menulis skrip demo.
    f.write(skrip_naif)

# Awal blok penanganan timeout.
try:
    # Menjalankan skrip dengan batas waktu 3 detik dan menangkap outputnya.
    hasil = subprocess.run([sys.executable], capture_output=True, text=True, timeout=3)
    # Dicetak jika skrip selesai normal.
    print("Output:", hasil.stdout)
    # Petunjuk untuk membaca output di atas.
    print("Putaran selesai semua? Lihat output di atas.")
# Menangkap timeout.
except subprocess.TimeoutExpired as e:
    # Mencetak bahwa solusi naif macet.
    print("TIMEOUT setelah 3 detik -- SOLUSI NAIF INI MACET (tidak menyelesaikan semua putaran)!")
    # Judul output parsial.
    print("Output sebelum macet:")
    # Menampilkan output sebelum macet: beberapa putaran berhasil, lalu berhenti pada pasangan yang
    # diperebutkan.
    print(e.stdout.decode() if e.stdout else "(kosong)")

### Solusi Benar: Agent Mengambil Keputusan Langsung (Tanpa Pusher Independen)

In [ ]:
# Memuat `threading`.
import threading
# Memuat `random` untuk memilih pasangan bahan acak.
import random
# Memuat `time`.
import time

# Target 8 putaran.
TARGET_PUTARAN = 8
# Lock yang melindungi keadaan meja (`di_meja`).
mutex_meja = threading.Lock()
# Status eksplisit: bahan mana yang sedang ada di meja (`True`/`False`).
di_meja = {"tembakau": False, "kertas": False, "korek": False}

# Sinyal khusus untuk perokok yang punya korek.
sinyal_perokok_korek = threading.Semaphore(0)     # dibangunkan jika tembakau+kertas tersedia (dia punya korek)
# Sinyal khusus untuk perokok yang punya tembakau.
sinyal_perokok_tembakau = threading.Semaphore(0)  # dibangunkan jika kertas+korek tersedia (dia punya tembakau)
# Sinyal khusus untuk perokok yang punya kertas.
sinyal_perokok_kertas = threading.Semaphore(0)    # dibangunkan jika tembakau+korek tersedia (dia punya kertas)
# Semaphore (awal 1): agent hanya boleh menaruh bahan baru bila perokok sebelumnya selesai.
agent_boleh_jalan = threading.Semaphore(1)
# Penghitung putaran selesai.
putaran_selesai = [0]

# Fungsi agent, satu-satunya pengambil keputusan.
def agent():
    # Mengulang sebanyak target.
    for _ in range(TARGET_PUTARAN):
        # Menunggu giliran.
        agent_boleh_jalan.acquire()
        # Memilih dua bahan acak.
        pilihan = random.choice(["tembakau_kertas", "kertas_korek", "tembakau_korek"])
        # Mencetak pilihan agent.
        print(f"[Agent] menaruh: {pilihan}")
        # Mengunci meja sehingga seluruh langkah di bawah bersifat atomik.
        with mutex_meja:
            # Jika tembakau + kertas...
            if pilihan == "tembakau_kertas":
                # ...tandai tembakau ada di meja...
                di_meja["tembakau"] = True
                # ...dan kertas juga.
                di_meja["kertas"] = True
            # Jika kertas + korek...
            elif pilihan == "kertas_korek":
                # ...kertas ada...
                di_meja["kertas"] = True
                # ...dan korek ada.
                di_meja["korek"] = True
            # Selain itu (tembakau + korek)...
            else:
                # ...tembakau ada...
                di_meja["tembakau"] = True
                # ...dan korek ada.
                di_meja["korek"] = True

            # Agent SENDIRI yang memutuskan dan langsung membangunkan perokok
            # yang tepat -- TIDAK ADA pusher terpisah yang bisa salah rebutan.
            # Agent sendiri memeriksa meja: tembakau + kertas lengkap?
            if di_meja["tembakau"] and di_meja["kertas"]:
                # Ambil bahan dari meja (reset ke `False`).
                di_meja["tembakau"] = di_meja["kertas"] = False
                # Bangunkan langsung perokok yang punya korek; tidak ada pusher yang bisa salah
                # tafsir.
                sinyal_perokok_korek.release()
            # Jika kertas + korek lengkap...
            elif di_meja["kertas"] and di_meja["korek"]:
                # Ambil bahan dari meja.
                di_meja["kertas"] = di_meja["korek"] = False
                # Bangunkan perokok yang punya tembakau.
                sinyal_perokok_tembakau.release()
            # Jika tembakau + korek lengkap...
            elif di_meja["tembakau"] and di_meja["korek"]:
                # Ambil bahan dari meja.
                di_meja["tembakau"] = di_meja["korek"] = False
                # Bangunkan perokok yang punya kertas.
                sinyal_perokok_kertas.release()

# Fungsi perokok dengan nama dan semaphore sinyalnya.
def perokok(nama, semaphore_sinyal):
    # Berulang sampai target tercapai.
    while putaran_selesai[0] < TARGET_PUTARAN:
        # Tidur sampai agent membangunkannya.
        semaphore_sinyal.acquire()
        # Mencetak bahwa perokok membuat dan menghisap rokok.
        print(f"  [{nama}] membuat & menghisap rokok")
        # Simulasi waktu merokok 20 ms.
        time.sleep(0.02)
        # Menambah hitungan putaran.
        putaran_selesai[0] += 1
        # Memberi izin agent menaruh bahan berikutnya.
        agent_boleh_jalan.release()

# Awal daftar thread: satu agent dan tiga perokok.
threads = [
    # Thread agent.
    threading.Thread(target=agent, daemon=True),
    # Perokok yang punya korek, dibangunkan oleh `sinyal_perokok_korek`.
    threading.Thread(target=perokok, args=("Perokok-Korek (punya tembakau+kertas)", sinyal_perokok_korek), daemon=True),
    # Perokok yang punya tembakau.
    threading.Thread(target=perokok, args=("Perokok-Tembakau (punya kertas+korek)", sinyal_perokok_tembakau), daemon=True),
    # Perokok yang punya kertas.
    threading.Thread(target=perokok, args=("Perokok-Kertas (punya tembakau+korek)", sinyal_perokok_kertas), daemon=True),
]
# Loop memulai thread.
for t in threads:
    # Menjalankan thread.
    t.start()
# Loop menunggu thread.
for t in threads:
    # Menunggu paling lama 5 detik per thread. Perokok masih tertahan di `acquire()` setelah target
    # tercapai (thread `daemon`), jadi `timeout` diperlukan agar program utama tidak menunggu
    # selamanya.
    t.join(timeout=5)

# Verifikasi: putaran selesai = target, ditulis BERHASIL atau GAGAL.
print(f"\nPutaran selesai: {putaran_selesai[0]} / {TARGET_PUTARAN} -- {'BERHASIL' if putaran_selesai[0]==TARGET_PUTARAN else 'GAGAL'}")

> 💡 **Pelajaran terpenting dari masalah ini:** tidak semua masalah sinkronisasi bisa diselesaikan dengan "menambahkan semaphore lagi" secara naif. Kadang solusi yang benar justru **menyederhanakan** desain — memusatkan pengambilan keputusan di **satu titik** (agent + mutex) daripada mendistribusikannya ke banyak thread independen yang berpotensi salah tafsir kondisi bersama.

## 6️⃣ Latihan Mandiri

Kerjakan latihan berikut sebagai bagian dari **Tugas Praktikum** (bobot 30%, dikumpulkan dalam bentuk notebook `.ipynb`):

1. **Dining Philosophers dengan N Berbeda:** Ulangi solusi arbitrator (bagian 1) dengan N = 3, 7, dan 10 filsuf. Ukur throughput (total makan per detik) untuk masing-masing N. Apakah throughput naik linear terhadap N? Jelaskan mengapa (atau mengapa tidak).

2. **Readers-Writers dengan Prioritas Writer:** Modifikasi solusi fair (bagian 2) menjadi solusi yang secara eksplisit **memprioritaskan writer** — begitu ada writer menunggu, TIDAK ADA reader baru yang boleh mulai membaca sampai writer tersebut selesai (reader yang sudah membaca boleh menyelesaikan bacaannya). Uji apakah reader baru sekarang yang berisiko starvation jika writer datang terus-menerus.

3. **Sleeping Barber dengan Banyak Tukang Cukur:** Modifikasi bagian 4 menjadi **2 tukang cukur** yang berbagi satu ruang tunggu yang sama. Pikirkan bagaimana menentukan tukang cukur mana yang melayani pelanggan berikutnya (hint: gunakan `Semaphore` tambahan atau `Queue` untuk mengelola giliran antar-tukang-cukur).

4. **Refleksi singkat (maks. 150 kata):** Dari kelima classical problem yang dibahas (Dining Philosophers, Readers-Writers, Producer-Consumer, Sleeping Barber, Cigarette Smokers), menurut kalian mana yang **paling mirip** dengan masalah nyata yang mungkin kalian temui dalam pengembangan software (web server, database, sistem embedded, dsb)? Berikan contoh konkret.

> 📤 **Pengumpulan:** Simpan sebagai `NIM_Nama_Tugas6.ipynb`, sertakan output dan penjelasan pada setiap sel (bukan hanya kode).

### ✅ Pembahasan & Jawaban Latihan Mandiri

#### Latihan 1: Simulasi Dining Philosophers dengan Variasi Jumlah Filsuf (N = 3, 7, 10)

**Strategi Penyelesaian.** Kode solusi arbitrator dari bagian pertama kita kemas ke dalam satu fungsi bernama `uji_arbitrator(jumlah_filsuf)` agar mudah dijalankan berulang kali untuk berbagai ukuran meja. Setiap skenario diuji dengan 20 siklus makan-berpikir per filsuf, durasi makan 0,02 detik, dan jeda berpikir 0,01 detik. Nilai **throughput** dihitung dari total aktivitas makan dibagi total waktu eksekusi. Nilai $N = 5$ dari percobaan sebelumnya juga tetap disertakan sebagai acuan pembanding bersama **batas maksimum teoretisnya**: mengingat satu filsuf memerlukan 2 garpu sekaligus dari total $N$ garpu yang ada, maksimal hanya $\lfloor N/2 \rfloor$ filsuf yang dapat makan pada saat bersamaan, sehingga kapasitas puncaknya berada di kisaran $\lfloor N/2 \rfloor \div 0,02$ makan per detik.

In [1]:
# Mengimpor modul threading untuk pembuatan thread, lock, dan semaphore.
import threading
# Mengimpor modul time untuk pengaturan jeda serta pengukuran durasi eksekusi.
import time

# Mengemas solusi arbitrator ke dalam fungsi agar pengujian bisa diulang untuk berbagai nilai N.
def uji_arbitrator(jumlah_filsuf, jml_siklus=20, durasi_makan=0.02, durasi_pikir=0.01):
    # Menyiapkan satu objek Lock untuk setiap garpu di meja.
    garpu = [threading.Lock() for _ in range(jumlah_filsuf)]
    # Membuat semaphore pelayan (arbitrator) yang membatasi maksimal N-1 filsuf mengambil garpu serentak.
    pelayan = threading.Semaphore(jumlah_filsuf - 1)
    # Menyimpan catatan berapa kali tiap filsuf sudah makan.
    hitungan_makan = [0] * jumlah_filsuf

    # Mendefinisikan rutinitas kerja untuk masing-masing thread filsuf.
    def aktivitas_filsuf(idx):
        # Menentukan garpu di sisi kiri sesuai indeks filsuf.
        garpu_kiri = garpu[idx]
        # Menentukan garpu di sisi kanan dengan operasi modulo agar posisi melingkar.
        garpu_kanan = garpu[(idx + 1) % jumlah_filsuf]
        # Menjalankan siklus makan dan berpikir sebanyak jml_siklus kali.
        for _ in range(jml_siklus):
            # Meminta izin duduk/ambil garpu ke pelayan (maksimal N-1 filsuf di dalam).
            with pelayan:
                # Mengunci garpu sebelah kiri terlebih dahulu.
                garpu_kiri.acquire()
                # Mengunci garpu sebelah kanan sebelum mulai makan.
                garpu_kanan.acquire()
                # Menambahkan catatan makan milik filsuf yang bersangkutan.
                hitungan_makan[idx] += 1
                # Mensimulasikan proses makan selama durasi_makan detik.
                time.sleep(durasi_makan)
                # Melepaskan kembali garpu kanan setelah selesai makan.
                garpu_kanan.release()
                # Melepaskan kembali garpu kiri agar bisa dipakai tetangganya.
                garpu_kiri.release()
            # Mensimulasikan fase berpikir tanpa memegang garpu apa pun.
            time.sleep(durasi_pikir)

    # Menyusun daftar thread untuk seluruh filsuf di meja.
    daftar_thread = [threading.Thread(target=aktivitas_filsuf, args=(idx,)) for idx in range(jumlah_filsuf)]
    # Mencatat titik waktu awal tepat sebelum seluruh thread dijalankan.
    waktu_mulai = time.perf_counter()
    # Menjalankan semua thread filsuf secara bersamaan.
    for th in daftar_thread:
        # Memulai eksekusi thread.
        th.start()
    # Menunggu seluruh thread filsuf menyelesaikan semua siklusnya.
    for th in daftar_thread:
        # Memblokir program utama sampai thread selesai.
        th.join()
    # Menghitung total waktu yang dihabiskan selama simulasi berlangsung.
    total_waktu = time.perf_counter() - waktu_mulai
    # Mengembalikan total porsi makan seluruh filsuf beserta durasi totalnya.
    return sum(hitungan_makan), total_waktu

# Menampilkan judul kolom tabel hasil pengukuran.
print(f"{'N':>3} | {'total makan':>11} | {'durasi (s)':>10} | {'throughput (makan/s)':>20} | {'batas teoretis':>14} | {'per filsuf':>10}")
# Mencetak garis pembatas tabel agar rapi.
print("-" * 89)
# Menyiapkan dictionary untuk menampung nilai throughput tiap skenario N.
data_throughput = {}
# Menguji skenario N = 3, 7, dan 10 sesuai instruksi, ditambah N = 5 sebagai acuan.
for n in [3, 5, 7, 10]:
    # Menjalankan simulasi untuk nilai n saat ini.
    total_makan, durasi_uji = uji_arbitrator(n)
    # Menghitung throughput sebagai rasio total makan dibagi waktu eksekusi.
    tp = total_makan / durasi_uji
    # Menghitung batas puncak teoretis: maksimal (n // 2) filsuf makan bareng, masing-masing 0,02 detik.
    batas_maks = (n // 2) / 0.02  # kapasitas teoretis dengan floor(n/2) filsuf paralel
    # Menyimpan hasil throughput ke dalam dictionary.
    data_throughput[n] = tp
    # Mencetak satu baris ringkasan metrik untuk nilai n tersebut.
    print(f"{n:>3} | {total_makan:>11} | {durasi_uji:>10.2f} | {tp:>20.1f} | {batas_maks:>14.1f} | {tp/n:>10.1f}")

# Memberi jarak satu baris kosong sebelum perbandingan rasio.
print()
# Menampilkan judul bagian perbandingan kenaikan throughput terhadap N = 3.
print("Perbandingan rasio throughput terhadap N=3:")
# Menghitung dan menampilkan faktor kelipatan untuk N = 5, 7, dan 10.
for n in [5, 7, 10]:
    # Membandingkan rasio kenaikan throughput dengan rasio kenaikan jumlah filsuf.
    print(f"  N={n:>2}: throughput x{data_throughput[n]/data_throughput[3]:.2f}  (kenaikan N x{n/3:.2f})")

  N | total makan | durasi (s) | throughput (makan/s) | batas teoretis | per filsuf
-----------------------------------------------------------------------------------------
  3 |          60 |       1.28 |                 47.0 |           50.0 |       15.7
  5 |         100 |       1.06 |                 93.9 |          100.0 |       18.8
  7 |         140 |       1.00 |                140.5 |          150.0 |       20.1
 10 |         200 |       0.86 |                233.6 |          250.0 |       23.4

Perbandingan rasio throughput terhadap N=3:
  N= 5: throughput x2.00  (kenaikan N x1.67)
  N= 7: throughput x2.99  (kenaikan N x2.33)
  N=10: throughput x4.97  (kenaikan N x3.33)


### Analisis Hasil Latihan 1

**Apakah throughput meningkat secara linear terhadap nilai N? Jawabannya tidak sepenuhnya.** Nilai throughput memang bertambah seiring membesarnya $N$, namun laju kenaikannya mengikuti pola **$\lfloor N/2 \rfloor$**, bukan berbanding lurus 1:1 dengan $N$:

- **Garpu menjadi faktor pembatas utama, bukan CPU.** Setiap filsuf membutuhkan dua garpu untuk makan. Akibatnya, dalam satu waktu paling banyak hanya $\lfloor N/2 \rfloor$ filsuf yang bisa makan secara bersamaan: untuk $N = 3$ hanya 1 filsuf, $N = 5$ maksimal 2 filsuf, $N = 7$ maksimal 3 filsuf, dan $N = 10$ maksimal 5 filsuf.
- **Hasil pengukuran mendekati batas teoretis.** Angka throughput yang tercatat di lapangan berada sangat dekat dengan batas maksimum teoretisnya (sekitar 96–99%). Selisih tipis ini wajar terjadi karena adanya jeda berpikir, biaya pembuatan *thread*, serta pergantian konteks (*context switching*) oleh sistem operasi. Hal ini sekaligus membuktikan bahwa semaphore arbitrator ($N - 1$ izin) **bukanlah penyebab kemacetan**, sebab batasan fisik $\lfloor N/2 \rfloor$ jauh lebih ketat.
- **Efisiensi per individu meningkat lalu melandai.** Throughput rata-rata per filsuf tidak bersifat konstan, melainkan naik dari $N = 3$ menuju $N = 10$ dan perlahan mendekati angka batas $1 \div (2 \times 0,02) = 25$ makan/detik per orang. Pada $N = 3$, hanya sepertiga (33%) filsuf yang bisa makan serentak; sedangkan pada $N = 10$, separuhnya (50%) bisa makan bersama. Proporsi filsuf yang aktif makan akan mendekati $\frac{1}{2}$ saat $N$ semakin besar, tetapi tidak akan pernah melampauinya.
- **Dampak jumlah $N$ ganjil.** Ketika $N$ bernilai ganjil, selalu ada satu garpu yang menganggur tanpa pasangan (misalnya pada $N = 7$, 3 filsuf memakai 6 garpu sehingga tersisa 1 garpu tak terpakai), membuat pemanfaatan sumber daya sedikit kurang optimal dibandingkan $N$ genap.
- **Waktu eksekusi keseluruhan justru semakin singkat.** Walaupun total porsi makan bertambah karena setiap filsuf tetap menjalankan 20 siklus, total durasi pengujian malah menurun seiring naiknya $N$ berkat meningkatnya jumlah filsuf yang dapat makan secara paralel.

**Intinya:** Besaran throughput mengikuti rumus $\approx \lfloor N/2 \rfloor \div \text{durasi\_makan}$. Pertumbuhannya memang linear terhadap $\lfloor N/2 \rfloor$ (dengan kemiringan sekitar $\frac{1}{2}$ per tambahan filsuf pada skala besar), tetapi **tidak naik secara proporsional 1:1 terhadap $N$**. Menambah jumlah filsuf ketika rasio garpu per orang tetap hanya menyediakan tambahan 1 garpu per kepala, sehingga kapasitas makan paralel hanya bertambah setiap kelipatan dua filsuf.

#### Latihan 2: Masalah Readers-Writers dengan Prioritas untuk Writer

In [2]:
# Mengimpor modul threading untuk sinkronisasi lock antar-thread.
import threading
# Mengimpor modul time untuk mensimulasikan durasi baca dan tulis.
import time

# ---- Implementasi PRIORITAS WRITER (Second Readers-Writers Problem, Courtois dkk. 1971) ----
# Menyimpan jumlah reader yang sedang aktif membaca.
jml_reader = 0
# Menyimpan jumlah writer yang sedang menunggu giliran maupun yang sedang menulis.
jml_writer = 0
# Lock untuk menjaga pembaruan variabel jml_reader agar aman dari race condition.
kunci_jml_reader = threading.Lock()     # pelindung akses ke jml_reader
# Lock untuk menjaga pembaruan variabel jml_writer agar tetap atomik.
kunci_jml_writer = threading.Lock()     # pelindung akses ke jml_writer
# Lock utama yang menjamin akses eksklusif ke data bersama.
akses_data = threading.Lock()           # pengunci eksklusif data bersama
# Gerbang masuk bagi reader; akan dikunci oleh writer pertama selama ada writer mengantre.
gerbang_reader = threading.Lock()       # pintu masuk reader yang ditutup saat writer menunggu

# Menyimpan data bersama yang diakses oleh reader dan writer.
shared_data = {"nilai": 0}
# Mencatat jumlah reader yang benar-benar berada di bagian kritis pembacaan.
reader_berjalan = [0]
# Mencatat jumlah writer yang benar-benar sedang menulis.
writer_berjalan = [0]
# Menghitung apabila terjadi pelanggaran aturan mutual exclusion (targetnya selalu 0).
jml_pelanggaran = [0]                   # penghitung pelanggaran sinkronisasi (harus 0)
# Lock pelindung untuk pencacah verifikasi di atas.
kunci_cek = threading.Lock()
# Menampung urutan kejadian untuk dicetak setelah seluruh thread selesai.
catatan_log = []
# Lock agar penulisan ke daftar catatan_log tidak bertabrakan.
kunci_log = threading.Lock()

# Fungsi pembantu untuk merekam peristiwa ke log secara aman.
def tulis_log(pesan):
    # Mengunci daftar log saat menambahkan pesan baru.
    with kunci_log:
        # Menyisipkan pesan ke akhir daftar.
        catatan_log.append(pesan)

# Rutinitas kerja untuk setiap thread reader.
def tugas_reader(rid):
    # Menggunakan variabel global jml_reader.
    global jml_reader
    # Reader wajib melewati gerbang_reader; jika ada writer menunggu, reader tertahan di baris ini.
    gerbang_reader.acquire()                    # cek apakah gerbang masuk reader terbuka
    # Mengunci penghitung reader sebelum mengubah nilainya.
    with kunci_jml_reader:
        # Menambah jumlah reader aktif.
        jml_reader += 1
        # Bila ini adalah reader pertama yang masuk...
        if jml_reader == 1:
            # ...kunci akses_data agar writer tidak bisa masuk selama masih ada yang membaca.
            akses_data.acquire()                # reader pertama memblokir akses writer
    # Segera melepas gerbang_reader agar antrean berikutnya tidak tertahan tanpa alasan.
    gerbang_reader.release()                    # buka kembali gerbang untuk thread lain

    # Memeriksa dan memperbarui status reader aktif saat mulai membaca.
    with kunci_cek:
        # Menambah pencacah reader yang sedang membaca.
        reader_berjalan[0] += 1
        # Jika ternyata ada writer aktif bersamaan, catat sebagai pelanggaran aturan.
        if writer_berjalan[0] > 0:
            # Menambah hitungan pelanggaran.
            jml_pelanggaran[0] += 1
    # Mencatat nilai data yang dibaca oleh reader saat ini.
    tulis_log(f"Reader-{rid} membaca nilai={shared_data['nilai']}")
    # Mensimulasikan waktu proses membaca selama 50 ms.
    time.sleep(0.05)
    # Memperbarui pencacah setelah reader selesai membaca.
    with kunci_cek:
        # Mengurangi jumlah reader yang sedang membaca.
        reader_berjalan[0] -= 1

    # Mengunci penghitung reader saat akan keluar.
    with kunci_jml_reader:
        # Mengurangi jumlah reader aktif.
        jml_reader -= 1
        # Bila ini adalah reader terakhir yang keluar...
        if jml_reader == 0:
            # ...bebaskan akses_data agar writer yang menunggu bisa segera masuk.
            akses_data.release()                # reader terakhir membuka akses untuk writer

# Rutinitas kerja untuk setiap thread writer.
def tugas_writer(wid):
    # Menggunakan variabel global jml_writer.
    global jml_writer
    # Mengunci penghitung writer saat mendaftarkan diri.
    with kunci_jml_writer:
        # Menambah jumlah writer yang menunggu atau aktif.
        jml_writer += 1
        # Jika ini writer pertama yang datang...
        if jml_writer == 1:
            # ...tutup gerbang_reader sehingga reader baru tidak boleh menyalip masuk.
            gerbang_reader.acquire()            # writer pertama menutup pintu bagi reader baru
    # Meminta akses eksklusif ke data (menunggu sampai reader yang terlanjur di dalam selesai).
    with akses_data:                            # masuk ke bagian kritis penulisan secara eksklusif
        # Memeriksa invarian keamanan sebelum menulis.
        with kunci_cek:
            # Menandai writer sedang aktif menulis.
            writer_berjalan[0] += 1
            # Pelanggaran terjadi jika ada lebih dari 1 writer atau masih ada reader aktif.
            if writer_berjalan[0] > 1 or reader_berjalan[0] > 0:
                # Mencatat pelanggaran yang terdeteksi.
                jml_pelanggaran[0] += 1
        # Memperbarui nilai pada data bersama (+1).
        shared_data["nilai"] += 1
        # Merekam aktivitas penulisan ke dalam log.
        tulis_log(f"Writer-{wid} menulis nilai baru={shared_data['nilai']}")
        # Mensimulasikan durasi operasi tulis selama 100 ms.
        time.sleep(0.1)
        # Memperbarui status setelah writer selesai menulis.
        with kunci_cek:
            # Menurunkan pencacah writer aktif.
            writer_berjalan[0] -= 1
    # Mengunci penghitung writer saat hendak keluar.
    with kunci_jml_writer:
        # Mengurangi jumlah writer yang menunggu/aktif.
        jml_writer -= 1
        # Jika sudah tidak ada lagi writer yang mengantre...
        if jml_writer == 0:
            # ...buka kembali gerbang_reader agar para reader bisa membaca lagi.
            gerbang_reader.release()            # writer terakhir membuka kembali pintu reader

# Pengujian fungsional: kedatangan bergelombang antara 6 reader dan 2 writer.
semua_thread = []
# Menambahkan 3 reader gelombang pertama (indeks 0–2).
for idx in range(3):
    # Membuat thread reader awal.
    semua_thread.append(threading.Thread(target=tugas_reader, args=(idx,)))
# Menambahkan 2 writer yang datang di tengah (indeks 0–1).
for idx in range(2):
    # Membuat thread writer.
    semua_thread.append(threading.Thread(target=tugas_writer, args=(idx,)))
# Menambahkan 3 reader gelombang kedua yang tiba setelah writer (indeks 3–5).
for idx in range(3, 6):
    # Membuat thread reader belakangan.
    semua_thread.append(threading.Thread(target=tugas_reader, args=(idx,)))

# Menjalankan semua thread dengan jeda tipis agar urutan kedatangannya terjaga.
for th in semua_thread:
    # Memulai eksekusi thread.
    th.start()
    # Memberi jeda 10 ms antar-kedatangan thread.
    time.sleep(0.01)
# Menunggu seluruh thread menyelesaikan tugasnya.
for th in semua_thread:
    # Memblokir hingga thread selesai.
    th.join()

# Menampilkan seluruh riwayat kejadian secara berurutan.
for baris in catatan_log:
    # Mencetak tiap entri log.
    print(baris)
# Memastikan nilai akhir data sama dengan 2 (hasil dari 2 kali penulisan).
print(f"\nNilai akhir: {shared_data['nilai']} (harus 2)")
# Memastikan tidak terjadi satu pun pelanggaran mutual exclusion.
print(f"Pelanggaran aturan (reader+writer bersamaan / 2 writer bersamaan): {jml_pelanggaran[0]} (harus 0)")

Reader-0 membaca nilai=0
Reader-1 membaca nilai=0
Reader-2 membaca nilai=0
Writer-0 menulis nilai baru=1
Writer-1 menulis nilai baru=2
Reader-3 membaca nilai=2
Reader-4 membaca nilai=2
Reader-5 membaca nilai=2

Nilai akhir: 2 (harus 2)
Pelanggaran aturan (reader+writer bersamaan / 2 writer bersamaan): 0 (harus 0)


**Pengujian Starvation pada Reader.** Di sini kita menjalankan empat *writer* yang terus menulis secara beruntun tanpa jeda, lalu menyisipkan satu *reader* di tengah jalan. Selama aktivitas para *writer* terus tumpang tindih, nilai `jml_writer` tidak pernah menyentuh angka 0 sehingga `gerbang_reader` tetap terkunci rapat.

In [3]:
# Mengimpor modul threading untuk simulasi konkurensi.
import threading
# Mengimpor modul time untuk pengukuran waktu tunggu.
import time

# Pengujian ekstrem: membuktikan apakah READER mengalami starvation bila writer datang tiada henti.
# Jumlah reader aktif saat ini.
jml_reader = 0
# Jumlah writer yang sedang mengantre atau menulis.
jml_writer = 0
# Lock pelindung jml_reader.
kunci_jml_reader = threading.Lock()
# Lock pelindung jml_writer.
kunci_jml_writer = threading.Lock()
# Lock eksklusif untuk akses data.
akses_data = threading.Lock()
# Gerbang masuk bagi reader.
gerbang_reader = threading.Lock()
# Event penanda bagi seluruh writer untuk menghentikan loop.
sinyal_berhenti = threading.Event()
# Menyimpan total operasi tulis yang berhasil dijalankan.
total_tulis = [0]
# Event penanda bahwa reader akhirnya berhasil masuk membaca.
reader_lolos = threading.Event()

# Fungsi writer yang terus-menerus menulis tanpa henti untuk mensimulasikan beban tulis padat.
def writer_tanpa_henti():
    # Mengacu pada variabel global jml_writer.
    global jml_writer
    # Terus berjalan selama sinyal_berhenti belum diaktifkan.
    while not sinyal_berhenti.is_set():
        # Mendaftarkan writer yang masuk antrean.
        with kunci_jml_writer:
            # Menambah hitungan writer.
            jml_writer += 1
            # Writer pertama segera menutup gerbang_reader.
            if jml_writer == 1:
                # Menutup akses masuk bagi reader baru.
                gerbang_reader.acquire()
        # Mengambil akses eksklusif untuk menulis.
        with akses_data:
            # Menambah total penulisan yang terjadi.
            total_tulis[0] += 1
            # Mensimulasikan proses tulis selama 5 ms.
            time.sleep(0.005)
        # Mengurangi hitungan writer setelah selesai menulis.
        with kunci_jml_writer:
            # Satu writer keluar.
            jml_writer -= 1
            # Gerbang reader hanya dibuka jika jml_writer benar-benar nol (jarang terjadi saat 4 writer tumpang tindih).
            if jml_writer == 0:
                # Membuka kembali gerbang masuk reader.
                gerbang_reader.release()

# Fungsi satu reader yang mencoba masuk di tengah padatnya arus writer.
def reader_tunggal():
    # Mengacu pada variabel global jml_reader.
    global jml_reader
    # Mencatat waktu mulai menunggu sebelum tertahan di gerbang.
    waktu_awal = time.perf_counter()
    # Mencoba melewati gerbang_reader (akan terblokir selama writer terus bergantian).
    gerbang_reader.acquire()
    # Mendaftarkan diri sebagai reader aktif.
    with kunci_jml_reader:
        # Menambah hitungan reader.
        jml_reader += 1
        # Reader pertama mengunci akses_data dari writer.
        if jml_reader == 1:
            # Mengambil lock data.
            akses_data.acquire()
    # Melepas gerbang_reader setelah berhasil masuk.
    gerbang_reader.release()
    # Menandai bahwa reader akhirnya berhasil mendapat giliran baca.
    reader_lolos.set()
    # Menampilkan total durasi tunggu reader hingga bisa membaca.
    print(f"Reader akhirnya dapat giliran setelah {time.perf_counter()-waktu_awal:.2f} detik menunggu")
    # Keluar dari bagian pembacaan.
    with kunci_jml_reader:
        # Mengurangi hitungan reader.
        jml_reader -= 1
        # Reader terakhir melepaskan lock data.
        if jml_reader == 0:
            # Membuka kembali akses_data.
            akses_data.release()

# Menyiapkan 4 thread writer yang bekerja terus-menerus.
grup_writer = [threading.Thread(target=writer_tanpa_henti) for _ in range(4)]
# Menjalankan seluruh thread writer.
for th in grup_writer:
    # Memulai thread writer.
    th.start()
# Memberi jeda 100 ms agar arus writer sudah padat lebih dulu.
time.sleep(0.1)                      # biarkan para writer mendominasi lebih awal
# Membuat thread reader tunggal sebagai daemon agar tidak menggantung jika macet.
th_reader = threading.Thread(target=reader_tunggal, daemon=True)
# Menjalankan thread reader.
th_reader.start()
# Menunggu reader maksimal selama 2 detik.
th_reader.join(timeout=2)
# Mengecek apakah reader masih belum berhasil masuk setelah batas waktu 2 detik.
if not reader_lolos.is_set():
    # Menampilkan bukti bahwa reader mengalami starvation.
    print("Reader BELUM juga dapat giliran setelah 2 detik -- READER STARVATION TERBUKTI!")
# Mengaktifkan sinyal berhenti agar semua writer keluar dari loop.
sinyal_berhenti.set()
# Menunggu seluruh thread writer berhenti sepenuhnya.
for th in grup_writer:
    # Memastikan thread writer selesai; setelah itu reader baru bisa lolos.
    th.join()
# Memberi kesempatan sebentar bagi thread reader untuk merampungkan eksekusinya.
th_reader.join(timeout=1)
# Menampilkan banyaknya operasi tulis yang berlangsung selama reader menunggu.
print("Total penulisan yang terjadi selama reader menunggu:", total_tulis[0])

Reader BELUM juga dapat giliran setelah 2 detik -- READER STARVATION TERBUKTI!
Reader akhirnya dapat giliran setelah 2.02 detik menunggu
Total penulisan yang terjadi selama reader menunggu: 391


### Analisis Hasil Latihan 2

**Apakah reader baru berpotensi mengalami starvation? Ya, terbukti terjadi.** Dalam simulasi pengujian di atas, *reader* yang datang belakangan sama sekali tidak kebagian giliran selama batas waktu penuh 2 detik, sementara ratusan operasi tulis terus berlangsung tanpa henti. *Reader* tersebut baru bisa masuk membaca setelah seluruh *writer* dipaksa berhenti lewat `sinyal_berhenti.set()`. Fenomena ini merupakan kebalikan persis dari *writer starvation* pada skema *reader-preference* di bagian 2—kali ini posisinya saja yang bertukar.

| Pendekatan Sinkronisasi | Pihak yang Diprioritaskan | Pihak yang Rentan Starvation |
|---|---|---|
| *Reader-preference* (bagian 2, awal) | *Reader* | ***Writer*** |
| *Fair* memakai pintu antrean bersama (bagian 2, akhir) | Urutan kedatangan (FIFO) | Tidak ada (*bounded waiting* terjamin) |
| *Writer-preference* (latihan ini) | *Writer* | ***Reader*** |

**Catatan tambahan:** Terjadinya *starvation* sangat **bergantung pada pola beban kerja sistem**. *Reader* hanya akan kelaparan apabila arus kedatangan *writer* terus-menerus bertumpuk tanpa henti. Jika sewaktu-waktu muncul jeda kosong tanpa ada *writer* yang mengantre, `gerbang_reader` akan langsung terbuka dan *reader* dapat masuk. Oleh karena itu, skema prioritas *writer* paling tepat diterapkan ketika kesegaran data menjadi prioritas utama (misalnya *writer* jarang muncul dan durasi tulisnya singkat). Sebaliknya, untuk beban lalu lintas campuran yang menuntut keadilan penuh bagi semua proses, pendekatan ***fair* (antrean FIFO)** tetap menjadi opsi paling aman.

#### Latihan 3: Simulasi Sleeping Barber dengan 2 Tukang Cukur

**Rancangan Solusi.** Dua orang tukang cukur melayani pelanggan secara bersamaan dengan berbagi **satu ruang tunggu** yang berkapasitas 3 kursi. Ada tiga keputusan utama dalam desain ini:

1. **Ruang tunggu dikelola sebagai antrean FIFO (`deque`)** yang dijaga oleh `kunci_ruang`. Dengan struktur ini, pelanggan yang tiba lebih awal dipastikan mendapat giliran cukur lebih dulu.
2. **Pembagian tugas antar-tukang-cukur menggunakan satu `Semaphore` bersama (`antrean_pelanggan`).** Tukang cukur mana saja yang berhasil memanggil `acquire()` lebih cepat akan langsung mengambil pelanggan di urutan paling depan, sedangkan tukang cukur yang tidak kebagian akan tidur menunggu kedatangan berikutnya. Mekanisme ini sudah cukup tanpa perlu membuat penjadwal terpisah.
3. **Notifikasi selesai dibuat spesifik untuk setiap pelanggan (`Event` di dalam objek `Klien`),** bukan memakai satu semaphore global seperti pada kasus 1 tukang cukur. Jika memakai semaphore global, sinyal selesai dari Tukang Cukur A bisa keliru ditangkap oleh pelanggan yang masih dicukur oleh Tukang Cukur B, sehingga pelanggan tersebut pergi sebelum cukurannya beres. Sinyal per individu memastikan hanya pelanggan yang bersangkutan yang dibangunkan.

Jumlah simulasi pelanggan ditingkatkan menjadi 14 orang dengan jeda kedatangan yang lebih rapat agar skenario penolakan saat kursi tunggu penuh dapat terlihat jelas.

In [4]:
# Mengimpor modul threading untuk mengelola thread tukang cukur dan pelanggan.
import threading
# Mengimpor modul time untuk simulasi durasi potong rambut dan jeda kedatangan.
import time
# Mengimpor modul random untuk menghasilkan interval kedatangan yang acak.
import random
# Mengimpor deque sebagai struktur antrean FIFO yang efisien untuk ruang tunggu.
from collections import deque

# Menetapkan jumlah tukang cukur yang bertugas bersamaan.
TOTAL_TUKANG = 2
# Menetapkan kapasitas maksimal kursi di ruang tunggu bersama.
KAPASITAS_KURSI = 3
# Menetapkan total pelanggan yang disimulasikan datang ke pangkas rambut.
TOTAL_PELANGGAN = 14
# Menetapkan durasi layanan cukur untuk satu pelanggan (dalam detik).
DURASI_CUKUR = 0.15

# Menyimpan daftar pelanggan yang sedang menunggu di kursi ruang tunggu secara FIFO.
kursi_tunggu = deque()                          # antrean FIFO pelanggan di ruang tunggu
# Lock pelindung untuk mengakses kursi_tunggu dan dictionary rekap secara aman.
kunci_ruang = threading.Lock()                  # menjaga atomisitas operasi ruang tunggu & statistik
# Semaphore penghitung pelanggan menunggu sekaligus penentu tukang cukur yang melayani berikutnya.
antrean_pelanggan = threading.Semaphore(0)      # bernilai 0 saat kosong sehingga tukang cukur tidur
# Menyimpan rekap jumlah layanan tiap tukang cukur serta total pelanggan yang pergi.
rekap = {"terlayani": [0] * TOTAL_TUKANG, "ditolak": 0}

# Lock agar output print dari berbagai thread tidak saling bertabrakan.
kunci_cetak = threading.Lock()
# Fungsi pembantu untuk mencetak pesan secara thread-safe.
def cetak_aman(pesan):
    # Mengunci akses layar/konsol selama proses cetak berlangsung.
    with kunci_cetak:
        # Menampilkan pesan ke konsol.
        print(pesan)

# Merepresentasikan setiap pelanggan agar memiliki penanda selesai masing-masing.
class Klien:
    # Menginisialisasi identitas pelanggan dan event notifikasi pribadinya.
    def __init__(self, nomor):
        # Menyimpan nomor urut kedatangan pelanggan.
        self.nomor = nomor
        # Event pribadi yang hanya di-set ketika pelanggan ini benar-benar selesai dicukur.
        self.sudah_selesai = threading.Event()  # sinyal selesai eksklusif milik pelanggan ini

# Rutinitas kerja untuk masing-masing thread tukang cukur.
def proses_tukang_cukur(no_tukang):
    # Tukang cukur terus bersiaga melayani selama program berjalan.
    while True:
        # Mengecek apakah ada pelanggan tanpa memblokir; jika kosong, tukang cukur bersiap tidur.
        if not antrean_pelanggan.acquire(blocking=False):   # cek cepat ketersediaan pelanggan
            # Mencatat status bahwa tukang cukur ini sedang tidur menunggu pelanggan.
            cetak_aman(f"[Tukang-{no_tukang}] tidur, menunggu pelanggan...")
            # Memblokir thread (tidur) hingga dibangunkan oleh sinyal pelanggan baru.
            antrean_pelanggan.acquire()                     # tidur sampai ada pelanggan masuk
        # Mengunci ruang tunggu untuk mengambil pelanggan di urutan terdepan.
        with kunci_ruang:
            # Mengeluarkan pelanggan paling depan sesuai prinsip FIFO.
            klien_aktif = kursi_tunggu.popleft()            # panggil pelanggan paling depan
            # Mencatat penambahan jumlah pelanggan yang ditangani tukang cukur ini.
            rekap["terlayani"][no_tukang] += 1
        # Mengumumkan bahwa tukang cukur mulai melayani pelanggan tersebut.
        cetak_aman(f"[Tukang-{no_tukang}] mencukur Pelanggan-{klien_aktif.nomor}...")
        # Mensimulasikan proses mencukur di luar lock agar tukang cukur lain tetap bisa bekerja paralel.
        time.sleep(DURASI_CUKUR)
        # Mengumumkan bahwa sesi cukur untuk pelanggan tersebut telah rampung.
        cetak_aman(f"[Tukang-{no_tukang}] selesai mencukur Pelanggan-{klien_aktif.nomor}")
        # Mengaktifkan event selesai milik pelanggan yang bersangkutan agar ia bisa pulang.
        klien_aktif.sudah_selesai.set()                     # beri sinyal selesai khusus ke klien ini

# Rutinitas kerja untuk setiap thread pelanggan yang datang.
def proses_pelanggan(no_pelanggan):
    # Membuat objek klien baru beserta event pribadinya.
    klien_baru = Klien(no_pelanggan)
    # Mengunci ruang tunggu untuk memeriksa ketersediaan kursi secara atomik.
    with kunci_ruang:
        # Memeriksa apakah jumlah pengantre masih di bawah kapasitas kursi tunggu.
        if len(kursi_tunggu) < KAPASITAS_KURSI:
            # Memasukkan pelanggan ke dalam antrean ruang tunggu.
            kursi_tunggu.append(klien_baru)
            # Mencatat bahwa pelanggan duduk beserta okupansi kursi saat ini.
            cetak_aman(f"[Pelanggan-{no_pelanggan}] duduk di ruang tunggu ({len(kursi_tunggu)}/{KAPASITAS_KURSI})")
            # Menandai bahwa pelanggan ini berhasil mendapat kursi dan akan menunggu dilayani.
            dapat_kursi = True
        # Jika seluruh kursi di ruang tunggu sudah penuh...
        else:
            # ...pelanggan langsung pergi meninggalkan tempat tanpa menunggu.
            cetak_aman(f"[Pelanggan-{no_pelanggan}] ruang tunggu PENUH, pergi!")
            # Menambah catatan statistik pelanggan yang ditolak karena penuh.
            rekap["ditolak"] += 1
            # Menandai bahwa pelanggan ini tidak dilayani.
            dapat_kursi = False
    # Melanjutkan proses menunggu hanya jika pelanggan berhasil memperoleh kursi.
    if dapat_kursi:
        # Menambah hitungan semaphore sekaligus membangunkan salah satu tukang cukur yang sedang tidur.
        antrean_pelanggan.release()             # beri sinyal kedatangan ke tukang cukur
        # Menunggu sampai sinyal selesai khusus milik pelanggan ini diaktifkan.
        klien_baru.sudah_selesai.wait()         # tunggu hingga cukuran dirinya selesai

# Membuat dan menjalankan seluruh thread tukang cukur sebagai daemon.
for idx in range(TOTAL_TUKANG):
    # Menjalankan thread tukang cukur ke-idx.
    threading.Thread(target=proses_tukang_cukur, args=(idx,), daemon=True).start()

# Mengatur seed acak agar pola kedatangan pelanggan konsisten saat diuji ulang.
random.seed(1)
# Menampung seluruh referensi thread pelanggan untuk keperluan join.
semua_pelanggan = []
# Mensimulasikan kedatangan pelanggan satu per satu.
for idx in range(TOTAL_PELANGGAN):
    # Memberi jeda acak 10–50 ms antar-kedatangan agar antrean sempat penuh.
    time.sleep(random.uniform(0.01, 0.05))
    # Membuat thread untuk pelanggan ke-idx.
    th = threading.Thread(target=proses_pelanggan, args=(idx,))
    # Memulai thread pelanggan.
    th.start()
    # Menyimpan thread ke dalam daftar.
    semua_pelanggan.append(th)

# Menunggu seluruh thread pelanggan selesai (baik yang terlayani maupun yang pergi).
for th in semua_pelanggan:
    # Memblokir sampai thread pelanggan selesai.
    th.join()

# Menampilkan judul bagian ringkasan hasil simulasi.
print("\n=== RINGKASAN ===")
# Menampilkan jumlah pelanggan yang ditangani oleh masing-masing tukang cukur.
for idx, jml in enumerate(rekap["terlayani"]):
    # Mencetak statistik layanan per tukang cukur.
    print(f"Tukang-{idx} melayani {jml} pelanggan")
# Menghitung total seluruh pelanggan yang berhasil dicukur.
total_terlayani = sum(rekap["terlayani"])
# Menampilkan jumlah pelanggan yang batal cukur karena ruang tunggu penuh.
print(f"Pelanggan pergi (penuh): {rekap['ditolak']}")
# Memverifikasi bahwa seluruh pelanggan tercatat lengkap tanpa ada yang hilang.
print(f"Dilayani + pergi = {total_terlayani + rekap['ditolak']} (harus {TOTAL_PELANGGAN})")

[Tukang-0] tidur, menunggu pelanggan...
[Tukang-1] tidur, menunggu pelanggan...
[Pelanggan-0] duduk di ruang tunggu (1/3)
[Tukang-0] mencukur Pelanggan-0...
[Pelanggan-1] duduk di ruang tunggu (1/3)
[Tukang-1] mencukur Pelanggan-1...
[Pelanggan-2] duduk di ruang tunggu (1/3)
[Pelanggan-3] duduk di ruang tunggu (2/3)
[Pelanggan-4] duduk di ruang tunggu (3/3)
[Tukang-0] selesai mencukur Pelanggan-0
[Tukang-0] mencukur Pelanggan-2...
[Pelanggan-5] duduk di ruang tunggu (3/3)
[Tukang-1] selesai mencukur Pelanggan-1
[Tukang-1] mencukur Pelanggan-3...
[Pelanggan-6] duduk di ruang tunggu (3/3)
[Pelanggan-7] ruang tunggu PENUH, pergi!
[Pelanggan-8] ruang tunggu PENUH, pergi!
[Pelanggan-9] ruang tunggu PENUH, pergi!
[Tukang-0] selesai mencukur Pelanggan-2
[Tukang-0] mencukur Pelanggan-4...
[Pelanggan-10] duduk di ruang tunggu (3/3)
[Tukang-1] selesai mencukur Pelanggan-3
[Tukang-1] mencukur Pelanggan-5...
[Pelanggan-11] duduk di ruang tunggu (3/3)
[Pelanggan-12] ruang tunggu PENUH, pergi!
[Pela

### Analisis Hasil Latihan 3

- **Eksekusi berjalan secara paralel.** Kedua tukang cukur melayani pelanggan berbarengan (tampak dari log aktivitas mencukur yang saling tumpang tindih) dengan pembagian beban kerja yang relatif seimbang pada ringkasan akhir.
- **Mekanisme penolakan bekerja saat kapasitas penuh.** Begitu seluruh kursi tunggu terisi, pelanggan baru yang datang langsung pulang tanpa mengantre—mirip cara kerja *connection pool* yang langsung menolak permintaan koneksi baru ketika seluruh slot sibuk. Pada bagian ringkasan, penjumlahan `terlayani + ditolak` selalu cocok dengan total pelanggan yang datang, membuktikan tidak ada pelanggan yang terlewat ataupun terlayani ganda.
- **Efisiensi CPU saat antrean kosong.** Tukang cukur yang mendapati ruang tunggu kosong otomatis masuk ke mode tidur (`acquire()` memblokir *thread*) tanpa membebani siklus CPU, lalu segera bangun begitu pelanggan berikutnya memanggil `release()`.
- **Kapasitas layanan meningkat dua kali lipat.** Secara teori, keberadaan dua tukang cukur menggandakan laju pelayanan menjadi $\approx 2 \div 0,15$ pelanggan per detik, sehingga tingkat penolakan pelanggan jauh lebih rendah dibandingkan skenario satu tukang cukur pada frekuensi kedatangan yang sama.
- **Penentuan tukang cukur bersifat non-deterministik.** Tukang cukur mana yang menangani pelanggan tertentu bergantung sepenuhnya pada penjadwalan *thread* sistem operasi saat keduanya memperebutkan semaphore. Meski begitu, sistem tetap menjamin urutan layanan sesuai antrean FIFO dan memastikan setiap pelanggan hanya dilayani tepat satu kali.

#### Latihan 4: Refleksi Singkat (maks. 150 kata)

> Menurut saya, pola yang paling sering dijumpai dalam pengembangan perangkat lunak nyata adalah **Producer-Consumer**. Pada arsitektur *web server*, lapisan penerima permintaan berperan sebagai *producer* yang memasukkan *request* ke dalam *message queue*, sementara kumpulan *worker thread* bertindak sebagai *consumer* yang mengeksekusinya. Pola serupa juga sangat umum di sistem *embedded*: rutin *interrupt* atau pembacaan sensor menyetor data ke *ring buffer*, lalu *loop* utama mengambil dan mengolahnya. Tantangan sinkronisasinya identik—struktur antrean harus aman dari akses bersamaan (*thread-safe*), *consumer* wajib menunggu saat *buffer* kosong, dan *producer* perlu menahan diri atau menolak beban ketika kapasitas penuh. Selain itu, **Sleeping Barber** juga sangat relevan untuk memodelkan *database connection pool* berkapasitas tetap yang menolak klien baru saat penuh, sedangkan **Readers-Writers** cerminan nyata dari mekanisme *caching* yang jauh lebih sering dibaca daripada diperbarui.